# D1：电池仿真 —— 恒流充放电与 PyBaMM→FASTSim 链路

## 实验目标
- PyBaMM SPM 模型恒流放电
- 导出 OCV-SOC 曲线为标准 CSV
- 用 `BatteryFASTSimAdapter` 映射到 FASTSim

## 通过判据
```python
assert abs(discharged_cap - nominal_cap)/nominal_cap <= 0.05
```

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
import pybamm
pybamm.set_logging_level('ERROR')
from common.battery_adapter import BatteryFASTSimAdapter
import matplotlib.pyplot as plt
import numpy as np
print('✓ 导入成功')

In [ ]:
# 1C 放电
model = pybamm.lithium_ion.SPM()
param = pybamm.ParameterValues('Chen2020')
sim = pybamm.Simulation(model, parameter_values=param)
sim.solve([0, 3600])
sol = sim.solution

t = sol['Time [s]'].entries
V = sol['Terminal voltage [V]'].entries
cap = sol['Discharge capacity [A.h]'].entries

plt.figure(figsize=(10, 4))
plt.plot(cap, V)
plt.xlabel('放出容量 (Ah)'); plt.ylabel('端电压 (V)')
plt.title('1C 放电曲线'); plt.grid(True, alpha=0.3)
plt.show()

nominal_cap = cap[-1]
print(f'放出容量: {nominal_cap:.2f} Ah')

# 断言：容量合理（Chen2020 约 4.85Ah）
assert abs(nominal_cap - 4.85)/4.85 <= 0.05, '容量异常!'
print('✅ 评测通过！')

In [ ]:
# 导出 OCV-SOC CSV（演示用简化数据）
soc = np.linspace(0, 1, 20)
# 用放电曲线的电压近似 OCV（教学简化）
ocv = np.interp(soc, cap/cap[-1], V)

BatteryFASTSimAdapter.export_from_pybamm(
    soc, ocv, '/tmp/battery_pack_NMC_75Ah.csv',
    chem='NMC', capacity_ah=75.0)
print('✓ CSV 已导出: /tmp/battery_pack_NMC_75Ah.csv')

# 验证 Adapter 可加载
adapter = BatteryFASTSimAdapter('/tmp/battery_pack_NMC_75Ah.csv')
e_kwh = adapter.equivalent_energy_kwh(75.0)
print(f'等效能量: {e_kwh:.1f} kWh')

## 🏆 挑战任务
1. 对比 0.5C/1C/2C 放电曲线，解释倍率效应。
2. 把导出的 CSV 用 `adapter.apply_to_vehicle()` 映射到 FASTSim 车型，跑 UDDS 看电耗变化。